# 07 · Guide depletion

A guide whose target is essential kills the cells that receive it, so the guide
is under-represented in the screen relative to the plasmid pool it was
delivered in. This notebook compares the two proportions per target gene.

The result is diagnostic: it does not filter anything. It is reported in the
supplementary figures and is worth reading before interpreting a knockout whose
cell count is low.

**Reads** `par_save_filename_5` and `par_initial_guide_pool_file`.
**Writes** `par_guide_depletion_file`.

`par_initial_guide_pool_file` has one row per guide and the number of cells
that guide contributed to the pool.

## Setup

In [1]:
from libraries import *
from parameters import *
from pathlib import Path

os.chdir(projectDir)
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.stats.multitest as smm

## Count cells per guide in the screen

Only cells carrying exactly one guide are counted, so that a cell is not
credited to two guides.

In [2]:
adata = sc.read(par_save_filename_5)
guide_names = list(adata.uns["feature_barcode_names"])

carried = (adata.obs[guide_names] > 0).astype(int)
single = carried[carried.sum(axis=1) == 1]
print(f"cells carrying exactly one guide: {single.shape[0]}")

screen_counts = single.sum(axis=0).rename("nCellsScreen").reset_index()
screen_counts.columns = ["Guide", "nCellsScreen"]
print(f"guides seen: {(screen_counts.nCellsScreen > 0).sum()} of {len(guide_names)}")

cells carrying exactly one guide: 341664
guides seen: 3716 of 3720


## Join the pool composition and aggregate to genes

Guide names in the pool table use `-` where the screen uses `_`; gene names
that legitimately contain a hyphen have to survive that substitution, so they
are restored explicitly.

In [3]:
pool = pd.read_csv(par_initial_guide_pool_file)
pool.columns = ["Guide", "nCellsPool"]
pool["nCellsPool"] = pool["nCellsPool"].astype(int)
pool["Guide"] = pool["Guide"].replace("-", "_", regex=True)

# The file carries a totals row in among the guides; it is not a guide.
pool = pool[pool.Guide != "Fullstats"].copy()

# gene names that really do contain a hyphen
for name in ["Rnf8-cmtr1", "Siah1-ps1", "Siah1-ps2"]:
    pool["Guide"] = pool["Guide"].str.replace(name.replace("-", "_"), name, regex=False)

missing = set(pool.Guide) - set(screen_counts.Guide)
if missing:
    print(f"in the pool but not the screen: {len(missing)}")

result = pd.merge(pool, screen_counts, on="Guide")
result["targetGene"] = ["_".join(g.split("_")[:-1]) for g in result.Guide]

per_gene = result.groupby("targetGene")[["nCellsPool", "nCellsScreen"]].sum()
per_gene = per_gene.drop(index=[par_not_target_control_prefix.rstrip("_"),
                                par_nongene_site_control_prefix.rstrip("_")],
                          errors="ignore")
print(f"target genes: {per_gene.shape[0]}")

target genes: 1130


## Test each gene for depletion

A one-sided two-proportion z-test: is the gene's share of screen cells smaller
than its share of pool cells?

In [4]:
total_screen = per_gene.nCellsScreen.sum()
total_pool = per_gene.nCellsPool.sum()

per_gene["nCellsPoolPerc"] = per_gene.nCellsPool / total_pool
per_gene["nCellsScreenPerc"] = per_gene.nCellsScreen / total_screen

pvals = []
for gene, row in per_gene.iterrows():
    _, p = proportions_ztest(
        count=np.array([row.nCellsScreen, row.nCellsPool]),
        nobs=np.array([total_screen, total_pool]),
        alternative="smaller",
    )
    pvals.append(p)

per_gene["pvalue"] = pvals
per_gene["FDR"] = smm.multipletests(per_gene.pvalue, method="fdr_bh")[1]
per_gene = per_gene.sort_values("FDR")

print(f"depleted at FDR < 0.1: {(per_gene.FDR < 0.1).sum()}")
print(per_gene.head(15).to_string())

depleted at FDR < 0.1: 419
               nCellsPool  nCellsScreen  nCellsPoolPerc  nCellsScreenPerc        pvalue           FDR
targetGene                                                                                           
Mdm2                 1880            30        0.001124          0.000101  1.240316e-61  1.401557e-58
Copa                 1896            72        0.001133          0.000242  5.682656e-46  3.210701e-43
Gnb4                 1356            26        0.000811          0.000087  3.317202e-43  1.249480e-40
Traip                1852            97        0.001107          0.000326  4.237450e-36  1.197080e-33
Cdc20                1232            34        0.000736          0.000114  2.686365e-35  6.071185e-33
Npm1                 1544            68        0.000923          0.000229  1.394641e-34  2.626574e-32
Map3k7               1656            86        0.000990          0.000289  1.024302e-32  1.653516e-30
Gm3701               1380            61        0.000825

## Write

In [5]:
Path(par_guide_depletion_file).parent.mkdir(parents=True, exist_ok=True)
per_gene.to_csv(par_guide_depletion_file)
print(f"written: {par_guide_depletion_file}")

written: TextFiles/NoOfCellsPerGuide_GeneLevel.csv
